# **03 - Patient-Level Summaries**

**Summary**

This notebook aggregated cleaned spell-level admissions into a patient-level dataset to support modelling of readmission risk and inequalities.
Each row now represents a unique patient, summarising utilisation, demographics, and key outcomes.

**Key observations**
- Length-of-stay and readmission distributions look realistic after rescaling.
- The dataset maintains diversity across deprivation quintiles and ethnic groups, supporting fairness analysis.
- No major missingness beyond IMD (< 2%), acceptable for downstream modelling.
- Aggregation successfully reduced multiple spells per patient, removing duplication bias.

**Output generated**
- `patient_level.parquet` - cleaned patient-level dataset ready for modelling.

In [1]:
import pandas as pd

In [2]:
# Read file
df = pd.read_parquet("../data/processed/apc_clean.parquet")

In [3]:
# Sort for readmission calculation
df = df.sort_values(["patient_id", "adm"])

# Calculate 30-day readmission flag
df["next_adm"] = df.groupby("patient_id")["adm"].shift(-1)
df["readmit_30d"] = (df["next_adm"] - df["dis"]).dt.days.between(1, 30)

# Aggregate to patient-level
patients = (
    df.groupby("patient_id")
      .agg(
          n_spells=("spell_id", "nunique"),
          mean_los=("los_days", "mean"),
          pct_emerg=("any_emerg", "mean"),
          readmit_30d=("readmit_30d", "max"),
          imd_quintile=("imd_quintile", "first"),
          sex=("sex", "first"),
          age=("age", "mean"),
          ethnicity_group=("ethnicity_group", "first"),
          respiratory_group_mode=("respiratory_group", lambda x: x.mode()[0] if not x.mode().empty else "Unknown")
      )
      .reset_index()
)

In [4]:
# Checks
print(f"Unique patients: {patients.shape[0]:,}")
print(f"Mean admissions per patient: {patients['n_spells'].mean():.2f}")
patients["readmit_30d"].value_counts(normalize=True)

Unique patients: 128,326
Mean admissions per patient: 1.13


readmit_30d
False    0.979404
True     0.020596
Name: proportion, dtype: float64

**Validation checks**

In [5]:
# Missingness
print(patients.isna().mean().sort_values(ascending=False))

imd_quintile              0.017572
patient_id                0.000000
n_spells                  0.000000
mean_los                  0.000000
pct_emerg                 0.000000
readmit_30d               0.000000
sex                       0.000000
age                       0.000000
ethnicity_group           0.000000
respiratory_group_mode    0.000000
dtype: float64


In [6]:
# LOS sanity
print(patients["mean_los"].describe())

count    128326.000000
mean          8.566268
std           8.953457
min           0.100000
25%           0.600000
50%           5.400000
75%          14.300000
max          36.500000
Name: mean_los, dtype: float64


In [7]:
# Check for outliers in n_spells
print(patients["n_spells"].value_counts().head(10))

n_spells
1     116679
2       8891
3       1672
4        520
5        224
6        120
7         72
8         66
9         37
10        24
Name: count, dtype: int64


In [8]:
# Confirm demographics coverage
print(patients["ethnicity_group"].value_counts(normalize=True))
print(patients["imd_quintile"].value_counts(normalize=True).sort_index())


ethnicity_group
White                     0.732026
Not stated                0.096325
Asian or Asian British    0.059045
Not known                 0.045610
Black or Black British    0.028544
Other Ethnic Groups       0.024742
Mixed                     0.013707
Name: proportion, dtype: float64
imd_quintile
1    0.222954
2    0.210754
3    0.200173
4    0.191321
5    0.174798
Name: proportion, dtype: Float64


In [9]:
# Save patient level summary to parquet
patients.to_parquet("../data/processed/patient_level.parquet")

**END**